In [15]:
import json
import numpy as np
import pandas as pd
import faiss
import importlib
import sys
import os

# Make src/ importable from notebooks/
sys.path.insert(0, os.path.abspath('..'))

print('All imports OK')

All imports OK


In [16]:
import src.index as idx_module
importlib.reload(idx_module)   # ensures any edits to src/index.py are picked up
from src.index import build_index, find_similar

build_index()

Index built: 1059 players, dim=128


In [17]:
# Verify the index
index = faiss.read_index('../models/faiss_index.bin')
print('Index type:     ', type(index).__name__)   # IndexFlatL2
print('Index dimension:', index.d)                # 64
print('Vectors stored: ', index.ntotal)           # should match player count

Index type:      IndexFlatL2
Index dimension: 128
Vectors stored:  1059


In [18]:
# Edit these to match your confirmed archetype names from the clustering notebook
archetype_names = {
     0: 'Shot-stopper',
     1: 'Generalist',
     2: 'Box-to-box engine',
     3: 'Possession system elite',
     4: 'Creative playmaker',
     5: 'Wide progressive',
     6: 'Defensive anchor',
     7: 'Aggressive stopper',
     8: 'Stopper CB',
     9: 'Attacking fullback',
    10: 'Aerial commander',
    -1: 'Uncategorised',
}

In [19]:
# Load canonical ordering — same row order as player_matrix.npy
player_ids = json.load(open('../models/player_ids.json'))
labels     = np.load('../models/labels.npy')

# Load metadata for name/position lookups only
meta = pd.read_csv('../data/processed/player_meta.csv')
meta['pid'] = meta['player_id'].astype(float).astype(int).astype(str)

id_to_name = dict(zip(meta['pid'], meta['name']))
id_to_pos  = dict(zip(meta['pid'], meta['position']))

# Build df — row index guaranteed to equal matrix row number
pid_strs = [str(int(float(p))) for p in player_ids]

df = pd.DataFrame({
    'player_id': pid_strs,
    'label':     labels,
    'archetype': [archetype_names.get(int(l), 'Uncategorised') for l in labels],
    'name':      [id_to_name.get(p, '—') for p in pid_strs],
    'position':  [id_to_pos.get(p,  '—') for p in pid_strs],
})

print(f'Players total:   {len(df)}')
print(f'Missing names:   {(df["name"] == "—").sum()}')
print(f'Uncategorised:   {(df["label"] == -1).sum()}')
df.head()

Players total:   1059
Missing names:   0
Uncategorised:   28


,player_id,label,archetype,name,position
0,5213,3,Possession system elite,Gerard Piqué Bernabéu,Right Center Back
1,10609,3,Possession system elite,Jérémy Mathieu,Left Center Back
2,5216,4,Creative playmaker,Andrés Iniesta Luján,Left Center Midfield
3,5503,3,Possession system elite,Lionel Andrés Messi Cuccittini,Right Wing
4,11179,0,Shot-stopper,Iker Casillas Fernández,Goalkeeper


In [20]:
# Alignment sanity check — must print 'Alignment OK: True'
test_name = 'Busquets'
match = df[df['name'].str.contains(test_name, case=False, na=False)]

if match.empty:
    print(f'{test_name} not found — try a different name')
else:
    idx         = match.index[0]
    pid_in_json = str(int(float(player_ids[idx])))
    pid_in_df   = df.iloc[idx]['player_id']
    print(f'df index:              {idx}')
    print(f'player_ids.json[{idx}]: {pid_in_json}')
    print(f'df player_id:          {pid_in_df}')
    print(f'Alignment OK:          {pid_in_json == pid_in_df}')   # must be True

df index:              23
player_ids.json[23]: 5203
df player_id:          5203
Alignment OK:          True


In [21]:
def search(name, top_k=10):
    """Find the top_k most similar players to `name` and return a DataFrame."""
    matches = df[df['name'].str.contains(name, case=False, na=False)]
    if matches.empty:
        print(f'Player "{name}" not found.')
        return None

    idx    = matches.index[0]
    target = df.iloc[idx]
    print(f'Query: {target["name"]} | {target["position"]} | {target["archetype"]}\n')

    results = find_similar(idx, top_k=top_k)
    rows = []
    for rank, (i, dist) in enumerate(results, 1):
        p = df.iloc[i]
        rows.append({
            'rank':      rank,
            'player':    p['name'],
            'position':  p['position'],
            'archetype': p['archetype'],
            'distance':  round(dist, 4),
        })
    result_df = pd.DataFrame(rows)
    print(result_df.to_string(index=False))
    return result_df

In [22]:
# Test 1 — defensive midfielder
search('Busquets')

Query: Sergio Busquets i Burgos | Center Defensive Midfield | Creative playmaker

 rank                         player                  position          archetype  distance
    1    James David Rodríguez Rubio                Right Wing Creative playmaker    0.0089
    2 Francisco José Beltrán Peinado Center Defensive Midfield Creative playmaker    0.0101
    3             Aleix Vidal Parreu                Right Back Creative playmaker    0.0121
    4   José Vicente Gómez Umpiérrez   Left Defensive Midfield Creative playmaker    0.0134
    5           Joan Verdú Fernández            Right Midfield Creative playmaker    0.0160
    6                   Ivan Rakitić     Right Center Midfield Creative playmaker    0.0166
    7            Alejandro Pozo Pozo                Right Back Creative playmaker    0.0169
    8      Jerónimo Figueroa Cabrera             Left Midfield Creative playmaker    0.0191
    9          Daniel Alves da Silva                Right Back Creative playmaker    0.019

,rank,player,position,archetype,distance
0,1,James David Rodríguez Rubio,Right Wing,Creative playmaker,0.0089
1,2,Francisco José Beltrán Peinado,Center Defensive Midfield,Creative playmaker,0.0101
2,3,Aleix Vidal Parreu,Right Back,Creative playmaker,0.0121
3,4,José Vicente Gómez Umpiérrez,Left Defensive Midfield,Creative playmaker,0.0134
4,5,Joan Verdú Fernández,Right Midfield,Creative playmaker,0.0160
5,6,Ivan Rakitić,Right Center Midfield,Creative playmaker,0.0166
6,7,Alejandro Pozo Pozo,Right Back,Creative playmaker,0.0169
7,8,Jerónimo Figueroa Cabrera,Left Midfield,Creative playmaker,0.0191
8,9,Daniel Alves da Silva,Right Back,Creative playmaker,0.0194
9,10,Foued Kadir,Right Wing,Creative playmaker,0.0233


In [23]:
# Test 2 — winger / creative
search('Messi')

Query: Lionel Andrés Messi Cuccittini | Right Wing | Possession system elite

 rank                              player                  position               archetype  distance
    1            Ronaldo de Assis Moreira                 Left Wing Possession system elite    0.0210
    2         Fábio Ricardo Gomes Fonseca                Right Wing Possession system elite    0.0397
    3                        Moussa Wagué                Right Back Possession system elite    0.0400
    4             Santiago Ezquerro Marín                 Left Wing Possession system elite    0.0520
    5                     Ousmane Dembélé                 Left Wing Possession system elite    0.0581
    6 Cristiano Ronaldo dos Santos Aveiro                 Left Wing Possession system elite    0.0592
    7              Iván Sánchez Rico Soto      Right Center Forward Possession system elite    0.0596
    8       Neymar da Silva Santos Junior                 Left Wing Possession system elite    0.0649
    

,rank,player,position,archetype,distance
0,1,Ronaldo de Assis Moreira,Left Wing,Possession system elite,0.0210
1,2,Fábio Ricardo Gomes Fonseca,Right Wing,Possession system elite,0.0397
2,3,Moussa Wagué,Right Back,Possession system elite,0.0400
3,4,Santiago Ezquerro Marín,Left Wing,Possession system elite,0.0520
4,5,Ousmane Dembélé,Left Wing,Possession system elite,0.0581
5,6,Cristiano Ronaldo dos Santos Aveiro,Left Wing,Possession system elite,0.0592
6,7,Iván Sánchez Rico Soto,Right Center Forward,Possession system elite,0.0596
7,8,Neymar da Silva Santos Junior,Left Wing,Possession system elite,0.0649
8,9,Fidel Chaves de la Torre,Left Wing,Possession system elite,0.0659
9,10,Yoshito Ōkubo,Center Attacking Midfield,Possession system elite,0.0665


In [24]:
# Test 3 — striker
search('Benzema')

Query: Karim Benzema | Center Forward | Creative playmaker

 rank                      player                  position          archetype  distance
    1 Samuel Chimerenka Chukwueze                Right Wing Creative playmaker    0.0094
    2    Luis Alberto Suárez Díaz            Center Forward Creative playmaker    0.0102
    3               Ludovic Giuly                Right Wing Creative playmaker    0.0134
    4   Yannick Ferreira Carrasco             Left Midfield Creative playmaker    0.0179
    5               Nabil El Zhar            Right Midfield Creative playmaker    0.0210
    6          Jorge Franco Alviz                 Left Wing Creative playmaker    0.0214
    7          Samuel Eto''o Fils            Center Forward Creative playmaker    0.0223
    8        Jonathan Viera Ramos                 Left Wing Creative playmaker    0.0226
    9 Carlos Alberto Vela Garrido                Right Wing Creative playmaker    0.0229
   10         Alberto Bueno Calvo Center Attacking

,rank,player,position,archetype,distance
0,1,Samuel Chimerenka Chukwueze,Right Wing,Creative playmaker,0.0094
1,2,Luis Alberto Suárez Díaz,Center Forward,Creative playmaker,0.0102
2,3,Ludovic Giuly,Right Wing,Creative playmaker,0.0134
3,4,Yannick Ferreira Carrasco,Left Midfield,Creative playmaker,0.0179
4,5,Nabil El Zhar,Right Midfield,Creative playmaker,0.0210
5,6,Jorge Franco Alviz,Left Wing,Creative playmaker,0.0214
6,7,Samuel Eto''o Fils,Center Forward,Creative playmaker,0.0223
7,8,Jonathan Viera Ramos,Left Wing,Creative playmaker,0.0226
8,9,Carlos Alberto Vela Garrido,Right Wing,Creative playmaker,0.0229
9,10,Alberto Bueno Calvo,Center Attacking Midfield,Creative playmaker,0.0239


In [25]:
# Test 4 — centre back
search('Piqué')

Query: Gerard Piqué Bernabéu | Right Center Back | Possession system elite

 rank                       player          position               archetype  distance
    1               Jérémy Mathieu  Left Center Back Possession system elite    0.0021
    2            Jean-Clair Todibo Right Center Back Possession system elite    0.0170
    3              Clément Lenglet  Left Center Back Possession system elite    0.0208
    4             Thomas Vermaelen  Left Center Back Possession system elite    0.0213
    5         Jesús Navas González        Right Back Possession system elite    0.0312
    6 Robin Aime Robert Le Normand  Left Center Back Possession system elite    0.0334
    7         Igor Zubeldia Elorza Right Center Back Possession system elite    0.0365
    8          Marc Bartra Aregall Right Center Back Possession system elite    0.0372
    9         Pau Francisco Torres  Left Center Back Possession system elite    0.0396
   10           Samuel Yves Umtiti  Left Center Back P

,rank,player,position,archetype,distance
0,1,Jérémy Mathieu,Left Center Back,Possession system elite,0.0021
1,2,Jean-Clair Todibo,Right Center Back,Possession system elite,0.0170
2,3,Clément Lenglet,Left Center Back,Possession system elite,0.0208
3,4,Thomas Vermaelen,Left Center Back,Possession system elite,0.0213
4,5,Jesús Navas González,Right Back,Possession system elite,0.0312
5,6,Robin Aime Robert Le Normand,Left Center Back,Possession system elite,0.0334
6,7,Igor Zubeldia Elorza,Right Center Back,Possession system elite,0.0365
7,8,Marc Bartra Aregall,Right Center Back,Possession system elite,0.0372
8,9,Pau Francisco Torres,Left Center Back,Possession system elite,0.0396
9,10,Samuel Yves Umtiti,Left Center Back,Possession system elite,0.0435


In [26]:
matrix          = np.load('../models/player_matrix.npy').astype('float32')
sample_indices  = np.random.choice(len(df), 20, replace=False)
index           = faiss.read_index('../models/faiss_index.bin')

dists, _  = index.search(matrix[sample_indices], 2)  # 2 = self + nearest neighbour
nn_dists  = np.sqrt(dists[:, 1])                     # sqrt: FAISS returns squared L2

print('Nearest-neighbour L2 distances (random sample of 20 players):')
print(f'  min:  {nn_dists.min():.5f}')
print(f'  max:  {nn_dists.max():.5f}')
print(f'  mean: {nn_dists.mean():.5f}')

Nearest-neighbour L2 distances (random sample of 20 players):
  min:  0.00957
  max:  0.21624
  mean: 0.10603


In [27]:
# Attach 2D UMAP coordinates for the style map scatter plot
coords_2d = np.load('../models/coords_2d.npy')
df['x']   = coords_2d[:, 0]
df['y']   = coords_2d[:, 1]

out_path = '../data/processed/player_meta_aligned.csv'
df.to_csv(out_path, index=True, index_label='matrix_row')

print(f'Saved {len(df)} rows to {out_path}')
print('Columns:', df.columns.tolist())
df.head()

Saved 1059 rows to ../data/processed/player_meta_aligned.csv
Columns: ['player_id', 'label', 'archetype', 'name', 'position', 'x', 'y']


,player_id,label,archetype,name,position,x,y
0,5213,3,Possession system elite,Gerard Piqué Bernabéu,Right Center Back,-0.562724,-3.027270
1,10609,3,Possession system elite,Jérémy Mathieu,Left Center Back,-0.512683,-3.062998
2,5216,4,Creative playmaker,Andrés Iniesta Luján,Left Center Midfield,0.675057,-2.547936
3,5503,3,Possession system elite,Lionel Andrés Messi Cuccittini,Right Wing,0.561460,-3.254647
4,11179,0,Shot-stopper,Iker Casillas Fernández,Goalkeeper,13.769799,6.334962


In [28]:
# What cluster is Messi actually in?
messi = df[df['name'].str.contains('Messi', case=False, na=False)].iloc[0]
print(f"Messi label: {messi['label']} | archetype: {messi['archetype']}")

# Who else is in that cluster?
cluster_members = df[df['label'] == messi['label']][['name', 'position', 'archetype']]
print(f"\nAll {len(cluster_members)} players in cluster {int(messi['label'])}:")
print(cluster_members.to_string())

Messi label: 3 | archetype: Possession system elite

All 79 players in cluster 3:
                                        name                   position                archetype
0                      Gerard Piqué Bernabéu          Right Center Back  Possession system elite
1                             Jérémy Mathieu           Left Center Back  Possession system elite
3             Lionel Andrés Messi Cuccittini                 Right Wing  Possession system elite
6                           Jordi Alba Ramos                  Left Back  Possession system elite
7               Kléper Laveran Lima Ferreira          Right Center Back  Possession system elite
8                        Sergio Ramos García           Left Center Back  Possession system elite
9                Javier Alejandro Mascherano           Left Center Back  Possession system elite
18            Marcelo Vieira da Silva Júnior                  Left Back  Possession system elite
19             Neymar da Silva Santos Junior 

<!-- 

With only **12 action tokens** the model cannot separate players who do the same *actions*  
in different *contexts*. Examples:

| Player | Real role | Why the model is unsure |
|---|---|---|
| Busquets | Deep-lying DM | High PASS_PROG count looks like a creative player |
| Full-backs | Wide defenders | High CARRY and PASS_CROSS overlaps with wingers |

These are documented limitations of a 12-token vocabulary — not bugs.  
Mention them in the README and note them as future improvements:
- Zone-aware tokens (`PASS_PROG_OWN_HALF` vs `PASS_PROG_OPP_HALF`)
- Sub-type carries (`CARRY_LONG` vs `CARRY_SHORT`)
- Richer vocabulary from StatsBomb 360 freeze-frame data -->